# Machine Learning + Technical Analysis on the S&P 500
### A pedagogical replication of Chin & Lin (2025) on a single-market universe

> Chin, J. T. and Lin, H. (2025). *Machine Learning and Technical Analysis in International Market.*
> Working paper, Victoria University of Wellington.

This notebook illustrates the central methodology of the paper on a tractable subset of the S&P 500.
The paper studies 25 international markets; we restrict to US large-caps so the data can be
sourced from Yahoo Finance. The headline question is the same:

**Do machine learning models trained on technical indicators alone produce economically meaningful
cross-sectional forecasts of stock returns, beyond what OLS can deliver?**

#### What we replicate
- Construction of ~107 technical features across 10 indicator families and 11 lookback windows
- Monthly cross-sectional forecasting of next-month returns
- Forecast combination across multiple ML models (Rapach, Strauss, Zhou 2010)
- Quintile long-short portfolio backtest, value-weighted and equal-weighted
- Out-of-sample $R^2$ vs. zero benchmark, with Clark-West (2007) significance test
- Carhart (1997) 4-factor alphas for risk adjustment
- SHAP feature importance (Lundberg & Lee 2017) on tree-based models

#### What we do *not* replicate (with reasoning)
- **Survivorship bias correction:** The paper uses both active and inactive companies via Refinitiv.
  Yahoo only serves currently-listed names. We use *current* S&P 500 constituents — this overstates
  realized returns. Treat absolute return numbers with caution; the ML-vs-OLS *gap* is more robust.
- **International coverage:** Single-market only; we cannot test the heterogeneity findings.
- **Feedforward neural networks:** Skipped for compute reasons; results in the paper show the
  forecast-combination already dominates individual models.
- **Bayesian hyperparameter optimization:** Replaced with sensible defaults / light grid search.
- **Long lookbacks (800, 1000 days):** Retained but require ~4 years of pre-sample data per stock.
- **Time-varying market-cap weighting:** Yahoo doesn't expose historical shares outstanding cleanly.
  We approximate VW using a rolling 60-day average of dollar volume as a liquidity proxy.

In [2]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from io import StringIO
import urllib.request
import zipfile

import yfinance as yf
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import lightgbm as lgb
import statsmodels.api as sm
import shap

np.random.seed(42)
plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

XGBoostError: 
XGBoost Library (libxgboost.dylib) could not be loaded.
Likely causes:
  * OpenMP runtime is not installed
    - vcomp140.dll or libgomp-1.dll for Windows
    - libomp.dylib for Mac OSX
    - libgomp.so for Linux and other UNIX-like OSes
    Mac OSX users: Run `brew install libomp` to install OpenMP runtime.

  * You are running 32-bit Python on a 64-bit OS

Error message(s): ["dlopen(/Users/yuriturygin/Documents/GitHub/statistical_arbitrage/.venv/lib/python3.14/site-packages/xgboost/lib/libxgboost.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib\n  Referenced from: <1A0D8152-BF46-3BE0-B651-EE965C187777> /Users/yuriturygin/Documents/GitHub/statistical_arbitrage/.venv/lib/python3.14/site-packages/xgboost/lib/libxgboost.dylib\n  Reason: tried: '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/V

## 1. Universe and data

We use the current S&P 500 constituents as our universe. To keep runtime manageable we pick
the top 80 by market capitalization. Daily OHLCV is fetched from Yahoo Finance.

The paper uses Jan 1996 – Nov 2021 with a 2007–2021 test window. We use Jan 2010 – Dec 2024
with a 2018–2024 test window — shorter than the paper's, but still spans the 2020 COVID crash
and the 2022 bear market, which lets us check the paper's claim that ML *outperforms more during
downturns*.

In [ ]:
# Hand-picked top S&P 500 names by market cap (as of 2024). Hardcoded to avoid
# scraping a constituent list, which is brittle.
TICKERS = [
    'AAPL','MSFT','NVDA','GOOGL','AMZN','META','BRK-B','TSLA','LLY','AVGO',
    'JPM','XOM','V','UNH','MA','PG','JNJ','HD','ORCL','COST',
    'MRK','ABBV','BAC','CVX','KO','WMT','PEP','ADBE','CRM','TMO',
    'MCD','ACN','LIN','CSCO','NFLX','ABT','AMD','DIS','WFC','DHR',
    'INTC','VZ','TXN','PM','NEE','IBM','CAT','GE','AMGN','UNP',
    'QCOM','PFE','LOW','RTX','SPGI','HON','BKNG','GS','BLK','UPS',
    'INTU','AXP','SBUX','LMT','SYK','GILD','BA','MDLZ','CVS','TJX',
    'ADP','VRTX','MMC','ISRG','PLD','REGN','SCHW','PYPL','MO','SO'
]

START_DATE = '2008-01-01'  # need 4 years pre-sample for 1000-day lags
END_DATE   = '2024-12-31'
TRAIN_END  = '2017-12-31'
VAL_START  = '2016-07-01'
VAL_END    = '2017-12-31'
TEST_START = '2018-01-01'

print(f'Universe: {len(TICKERS)} tickers')
print(f'Sample : {START_DATE} -> {END_DATE}')
print(f'Train  : {START_DATE} -> {TRAIN_END}')
print(f'Validation: {VAL_START} -> {VAL_END} (within train window)')
print(f'Test   : {TEST_START} -> {END_DATE}')

In [ ]:
# Bulk download. Yahoo returns OHLCV with adjustments for splits/dividends.
# yfinance occasionally hiccups on a few tickers; we handle missing names gracefully.
print('Downloading from Yahoo Finance (this can take ~1-3 minutes)...')
raw = yf.download(
    TICKERS, start=START_DATE, end=END_DATE,
    auto_adjust=True, progress=False, threads=True, group_by='ticker'
)

# Reorganize into a tidy panel: (date, ticker) -> ohlcv columns
frames = []
top_levels = list(raw.columns.get_level_values(0).unique()) if isinstance(raw.columns, pd.MultiIndex) else []
for tk in TICKERS:
    if tk not in top_levels:
        continue
    df = raw[tk].copy()
    if df.dropna(how='all').empty:
        continue
    # Standardize column names — yfinance returns 'Open','High','Low','Close','Volume'
    df.columns = [str(c).lower() for c in df.columns]
    needed = {'open','high','low','close','volume'}
    if not needed.issubset(df.columns):
        continue
    df = df[['open','high','low','close','volume']].copy()
    df['ticker'] = tk
    frames.append(df.reset_index().rename(columns={'Date':'date'}))

if not frames:
    raise RuntimeError(
        'No data returned from Yahoo. Possible causes: rate limiting, '
        'network issue, or yfinance API changes. Re-running often helps.'
    )

panel = pd.concat(frames, ignore_index=True)
panel.columns = [c.lower() if isinstance(c, str) else c for c in panel.columns]
panel = panel[['date','ticker','open','high','low','close','volume']].dropna()
panel['date'] = pd.to_datetime(panel['date'])
panel = panel.sort_values(['ticker','date']).reset_index(drop=True)

print(f'\nTickers retained: {panel.ticker.nunique()} / {len(TICKERS)}')
print(f'Total rows      : {len(panel):,}')
print(f'Date range      : {panel.date.min().date()} -> {panel.date.max().date()}')
panel.head()

## 2. Feature engineering — the 107 technical indicators

Following Section 3.1 of the paper, we construct **10 indicator families** evaluated at
**11 lookback windows** (3, 5, 10, 20, 50, 100, 200, 400, 600, 800, 1000 days):

**Trend**
- **SMA(n)** — simple moving average of close, normalized by current close: $\text{SMA}_n / P_t - 1$
- **EMA(n)** — exponential moving average of close, normalized by current close
- **ADX(n)** — Wilder's Average Directional Index; smoothed +DI/−DI

**Momentum**
- **ROC(n)** — rate of change: $P_t / P_{t-n} - 1$
- **RSI(n)** — relative strength index over n-period gains/losses, ∈ [0,100]
- **WILLR(n)** — Williams %R: position of close in n-day high-low range, ∈ [−100, 0]
- **CCI(n)** — Commodity Channel Index using typical price = (H+L+C)/3

**Volatility**
- **ATR(n)** — Wilder's Average True Range, normalized by close

**Volume / Money Flow**
- **PVT(n)** — n-day differenced Price-Volume Trend (cumulative volume×ROC, then differenced)
- **CMF(n)** — Chaikin Money Flow: volume-weighted average of close location in HL range

The paper normalizes SMA/EMA by the most recent price (Han, Zhou, Zhu 2016) so they become
deviation ratios rather than levels. We also normalize ATR by close to make it a percentage
volatility, which is the convention.

> **Multicollinearity note.** Adjacent-lag versions of the same indicator are mechanically
> near-duplicates (SMA-200 and SMA-400 share 200 of their input days). This is exactly the
> setting where OLS struggles — the design matrix is near-singular — and where ML's implicit
> dimensionality reduction (regularization in Elastic Net, feature selection in trees) earns
> its keep.

> **Feature count.** We generate the full 10 × 11 = **110** grid; the paper reports 107
> (likely some indicator-lag combinations are dropped where the math breaks down — e.g. ADX
> at very short lookbacks). The 3-feature difference is immaterial.

In [ ]:
LAGS = [3, 5, 10, 20, 50, 100, 200, 400, 600, 800, 1000]

def true_range(h, l, c):
    """True Range: max(H-L, |H-C_prev|, |L-C_prev|)."""
    pc = c.shift(1)
    return pd.concat([h - l, (h - pc).abs(), (l - pc).abs()], axis=1).max(axis=1)

def wilder_smooth(series, n):
    """Wilder's smoothing (used in ATR, RSI, ADX). Equivalent to EWMA with alpha=1/n."""
    return series.ewm(alpha=1.0/n, adjust=False).mean()

def compute_features(df):
    """
    Compute all ~110 technical indicators for a single stock's daily price history.
    Input: df with columns [open, high, low, close, volume], indexed by date, sorted ascending.
    Output: DataFrame of features indexed by date.
    """
    o, h, l, c, v = df.open, df.high, df.low, df.close, df.volume
    out = {}

    # ---- TREND ----
    for n in LAGS:
        out[f"SMA_{n}"] = c.rolling(n).mean() / c - 1.0
        out[f"EMA_{n}"] = c.ewm(span=n, adjust=False).mean() / c - 1.0

    # ADX needs +DM, -DM, ATR. Standard Wilder formulation.
    up_move = h.diff()
    dn_move = -l.diff()
    plus_dm = np.where((up_move > dn_move) & (up_move > 0), up_move, 0.0)
    mins_dm = np.where((dn_move > up_move) & (dn_move > 0), dn_move, 0.0)
    plus_dm = pd.Series(plus_dm, index=c.index)
    mins_dm = pd.Series(mins_dm, index=c.index)
    tr = true_range(h, l, c)
    for n in LAGS:
        atr_n = wilder_smooth(tr, n)
        plus_di = 100 * wilder_smooth(plus_dm, n) / atr_n.replace(0, np.nan)
        mins_di = 100 * wilder_smooth(mins_dm, n) / atr_n.replace(0, np.nan)
        dx = 100 * (plus_di - mins_di).abs() / (plus_di + mins_di).replace(0, np.nan)
        out[f"ADX_{n}"] = wilder_smooth(dx, n)

    # ---- MOMENTUM ----
    for n in LAGS:
        out[f"ROC_{n}"] = c / c.shift(n) - 1.0
        diff = c.diff()
        gain = diff.clip(lower=0)
        loss = -diff.clip(upper=0)
        avg_gain = wilder_smooth(gain, n)
        avg_loss = wilder_smooth(loss, n)
        rs = avg_gain / avg_loss.replace(0, np.nan)
        out[f"RSI_{n}"] = 100 - 100 / (1 + rs)

        hh = h.rolling(n).max()
        ll = l.rolling(n).min()
        out[f"WILLR_{n}"] = -100 * (hh - c) / (hh - ll).replace(0, np.nan)

        tp = (h + l + c) / 3.0
        sma_tp = tp.rolling(n).mean()
        mad = (tp - sma_tp).abs().rolling(n).mean()
        out[f"CCI_{n}"] = (tp - sma_tp) / (0.015 * mad.replace(0, np.nan))

    # ---- VOLATILITY ----
    for n in LAGS:
        out[f"ATR_{n}"] = wilder_smooth(tr, n) / c  # normalized

    # ---- VOLUME / MONEY FLOW ----
    pvt_inc = v * c.pct_change()
    pvt_cum = pvt_inc.cumsum()
    for n in LAGS:
        out[f"PVT_{n}"] = (pvt_cum - pvt_cum.shift(n)) / (v.rolling(n).sum().replace(0, np.nan))
        loc = ((c - l) - (h - c)) / (h - l).replace(0, np.nan)
        mfv = loc * v
        out[f"CMF_{n}"] = mfv.rolling(n).sum() / v.rolling(n).sum().replace(0, np.nan)

    return pd.DataFrame(out, index=df.index)

# Compute features per stock and stack
print("Computing features per ticker...")
all_feat = []
for tk, g in panel.groupby("ticker"):
    g2 = g.set_index("date").sort_index()
    if len(g2) < 1100:  # need >1000 days plus buffer
        continue
    f = compute_features(g2)
    f["ticker"] = tk
    f["close"]  = g2.close
    f["volume"] = g2.volume
    all_feat.append(f.reset_index())

features = pd.concat(all_feat, ignore_index=True)
feat_cols = [c for c in features.columns if c not in ("date","ticker","close","volume")]
print(f"Feature count : {len(feat_cols)}")
print(f"Panel rows    : {len(features):,}")
print(f"Tickers       : {features.ticker.nunique()}")

## 3. Monthly sampling and target construction

The paper computes features from daily data but predicts at **monthly** frequency. So we
sample each feature series at month-end and pair it with the **next month's simple return**
as the target. This creates non-overlapping monthly observations — there's no overlapping-returns
issue at the 1-month horizon.

Note that the *features* themselves remain highly autocorrelated month-to-month (SMA-1000 in
January and February share 979 of 1000 input days), even though the *target* sequence isn't.
This is the reason for using a clean time-based train/test split rather than random k-fold.

In [ ]:
# Pick the last trading day of each month per ticker
features['ym'] = features['date'].dt.to_period('M')
month_end_idx = features.groupby(['ticker','ym'])['date'].idxmax()
monthly = features.loc[month_end_idx].copy()

# Next-month simple return
monthly = monthly.sort_values(['ticker','date']).reset_index(drop=True)
monthly['close_next'] = monthly.groupby('ticker')['close'].shift(-1)
monthly['ret_fwd_1m'] = monthly['close_next'] / monthly['close'] - 1.0
monthly = monthly.dropna(subset=['ret_fwd_1m'])

# Drop rows where any feature is NaN (long lookbacks chew through early periods)
monthly = monthly.dropna(subset=feat_cols).reset_index(drop=True)

# Winsorize forward returns at top/bottom 1% within each month, per the paper
def winsorize_xs(s, lo=0.01, hi=0.99):
    ql, qh = s.quantile(lo), s.quantile(hi)
    return s.clip(ql, qh)

monthly['ret_fwd_1m'] = (
    monthly.groupby('date')['ret_fwd_1m']
           .transform(winsorize_xs)
)

print(f'Monthly panel rows: {len(monthly):,}')
print(f'Months : {monthly.date.nunique()}')
print(f'Tickers: {monthly.ticker.nunique()}')
print(f'Date range: {monthly.date.min().date()} -> {monthly.date.max().date()}')

## 4. Train/test split and standardization

Following the paper (Section 3.1), we standardize each feature to zero mean and unit variance
using **only training-period statistics**, then apply the same scaler to test data. This avoids
look-ahead bias.

Cutoffs:
- **Train:** through 2017-12-31
- **Test:** 2018-01-01 onward

In [ ]:
train_mask = monthly['date'] <= TRAIN_END
test_mask  = monthly['date'] >= TEST_START

X_tr = monthly.loc[train_mask, feat_cols].astype(float)
y_tr = monthly.loc[train_mask, 'ret_fwd_1m'].astype(float)
X_te = monthly.loc[test_mask , feat_cols].astype(float)
y_te = monthly.loc[test_mask , 'ret_fwd_1m'].astype(float)

# Replace inf/nan from any pathological feature value
X_tr = X_tr.replace([np.inf,-np.inf], np.nan).fillna(0)
X_te = X_te.replace([np.inf,-np.inf], np.nan).fillna(0)

scaler = StandardScaler().fit(X_tr.values)
X_tr_s = scaler.transform(X_tr.values)
X_te_s = scaler.transform(X_te.values)

print(f'Train obs: {len(X_tr):,}  ({y_tr.shape[0]} stock-months)')
print(f'Test  obs: {len(X_te):,}')
print(f'Features : {X_tr.shape[1]}')

## 5. Models

We fit five forecasting models and a forecast-combination ensemble:

| Model | Role in paper |
|---|---|
| OLS | benchmark linear |
| Elastic Net | regularized linear (Zou & Hastie 2005) |
| Random Forest | tree ensemble (Ho 1995) |
| XGBoost | gradient boosting |
| LightGBM | gradient boosting |
| **Combo** | simple average of forecasts (Rapach, Strauss, Zhou 2010) |

The paper does Bayesian hyperparameter optimization over a >10,000-combination search space.
We use sensible defaults — this is a known simplification and likely **understates** the ML
edge, since less-tuned ML models still tend to beat OLS but by a smaller margin.

In [ ]:
def fit_predict_models(X_tr, y_tr, X_te):
    """Fit each model on (X_tr, y_tr) and return a dict of test predictions."""
    preds = {}

    # OLS
    m = LinearRegression().fit(X_tr, y_tr)
    preds["OLS"] = m.predict(X_te)

    # Elastic Net (alpha and l1_ratio chosen as defaults; not tuned)
    m = ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=20000, random_state=42)
    m.fit(X_tr, y_tr)
    preds["ENet"] = m.predict(X_te)

    # Random Forest
    m = RandomForestRegressor(
        n_estimators=300, max_depth=6, min_samples_leaf=50,
        n_jobs=-1, random_state=42
    ).fit(X_tr, y_tr)
    preds["RF"] = m.predict(X_te)

    # XGBoost
    m = xgb.XGBRegressor(
        n_estimators=400, max_depth=4, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.6, reg_lambda=1.0,
        n_jobs=-1, random_state=42, verbosity=0
    ).fit(X_tr, y_tr)
    preds["XGB"] = m.predict(X_te)

    # LightGBM
    m = lgb.LGBMRegressor(
        n_estimators=400, max_depth=-1, num_leaves=31, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.6, reg_lambda=1.0,
        n_jobs=-1, random_state=42, verbosity=-1
    ).fit(X_tr, y_tr)
    preds["LGBM"] = m.predict(X_te)

    return preds, m  # return last model handle for SHAP later

print("Training models...")
preds, lgbm_model = fit_predict_models(X_tr_s, y_tr.values, X_te_s)

# Forecast combination
preds["Combo"] = np.mean([preds[k] for k in ["ENet","RF","XGB","LGBM"]], axis=0)

print("Done.")
for k, v in preds.items():
    print(f"  {k:6s}  mean={v.mean():+.4f}  std={v.std():.4f}")

## 6. Statistical significance — $R^2_{OS}$ and Clark-West

Following Eq. (3) of the paper, $R^2_{OS}$ is computed against a **zero-return benchmark**:

$$R^2_{OS} = 1 - \frac{\sum (r_{i,t+1} - \hat r_{i,t+1})^2}{\sum r_{i,t+1}^2}$$

A value above zero means the model beats predicting zero. Since most stock-month returns are
near zero, this is a low bar in absolute terms but a useful relative one.

The **Clark & West (2007) MSPE-adjusted statistic** tests whether one model's MSPE is
significantly smaller than another's. We compare every ML model to OLS, with the test
statistic $\bar f_t / \text{SE}(\bar f_t)$ where

$$f_t = (r_{t+1} - \hat r_{t+1}^{\text{OLS}})^2 - \big[(r_{t+1} - \hat r_{t+1}^{\text{ML}})^2 - (\hat r_{t+1}^{\text{OLS}} - \hat r_{t+1}^{\text{ML}})^2\big]$$

Standard errors use Newey-West with 6 lags, also per the paper.

> **Expected behavior:** OLS will likely post a *very* negative $R^2_{OS}$ (often worse than
> -50% in practice). This is not a bug — it's the multicollinearity blowup we discussed:
> with 110 highly correlated features and a non-singular but ill-conditioned design matrix,
> OLS coefficients are unstable and amplify noise out-of-sample. ML models, by contrast,
> should sit much closer to zero. This $R^2_{OS}$ gap is one of the cleanest demonstrations
> of why the paper's methodology requires ML rather than OLS as the reference forecaster.

In [ ]:
def r2_os(y, yhat):
    return 1.0 - np.sum((y - yhat)**2) / np.sum(y**2)

def clark_west(y, yhat_bench, yhat_alt, lags=6):
    """
    One-sided Clark-West test: H0 yhat_bench MSPE <= yhat_alt MSPE.
    Positive t-stat with low p-value means the alternative model has lower MSPE.
    """
    y = np.asarray(y); yhat_bench = np.asarray(yhat_bench); yhat_alt = np.asarray(yhat_alt)
    f = (y - yhat_bench)**2 - ((y - yhat_alt)**2 - (yhat_bench - yhat_alt)**2)
    fbar = f.mean()
    # Newey-West SE on f
    n = len(f); resid = f - fbar
    var = (resid**2).sum() / n
    for L in range(1, lags+1):
        w = 1 - L/(lags+1)
        cov = (resid[L:] * resid[:-L]).sum() / n
        var += 2 * w * cov
    se = np.sqrt(var / n)
    t = fbar / se
    from scipy.stats import norm
    pval = 1 - norm.cdf(t)  # one-sided
    return t, pval

rows = []
for name, yhat in preds.items():
    t, p = (np.nan, np.nan) if name == "OLS" else clark_west(y_te.values, preds["OLS"], yhat)
    rows.append({
        "model": name,
        "R2_OS (%)": 100 * r2_os(y_te.values, yhat),
        "CW t-stat vs OLS": t,
        "CW p-value": p,
    })
res_stat = pd.DataFrame(rows).round(4)
res_stat

## 7. Quintile long-short portfolio backtest

For each test month, we sort stocks into quintiles based on their predicted return and form a
**long-short portfolio**: long the top quintile (Q5), short the bottom quintile (Q1). Per the
paper Section 2.3, the portfolio is rebalanced monthly.

Two weighting schemes:
- **Equal-weighted** within each leg (per stock)
- **Value-weighted proxy** using a 60-day average dollar-volume weighting (since Yahoo doesn't give us historical shares-out)

The paper finds VW results are most economically meaningful (Fama 1998); we report both.

In [ ]:
# Attach predictions back to the test panel
test_panel = monthly.loc[test_mask, ["date","ticker","close","volume","ret_fwd_1m"]].copy()
test_panel = test_panel.reset_index(drop=True)
for name, yhat in preds.items():
    test_panel[f"yhat_{name}"] = yhat

# Approximate value-weight: trailing 60-day dollar volume captured at formation date
tmp = features[["date","ticker","close","volume"]].copy()
tmp["dv"] = tmp.close * tmp.volume
tmp = tmp.sort_values(["ticker","date"])
tmp["adv60"] = tmp.groupby("ticker")["dv"].transform(lambda s: s.rolling(60).mean())
dv = tmp[["date","ticker","adv60"]]
test_panel = test_panel.merge(dv, on=["ticker","date"], how="left")
test_panel["adv60"] = test_panel["adv60"].fillna(test_panel["adv60"].median())

def long_short_returns(df, signal_col, weight="equal"):
    """Form quintile long-short returns from a signal."""
    rows = []
    for dt, g in df.groupby("date"):
        if len(g) < 10:
            continue
        q = pd.qcut(g[signal_col], 5, labels=False, duplicates="drop")
        g = g.assign(q=q)
        if weight == "equal":
            top = g.loc[g.q == 4, "ret_fwd_1m"].mean()
            bot = g.loc[g.q == 0, "ret_fwd_1m"].mean()
        else:  # value-weight by ADV
            tg = g.loc[g.q == 4]; bg = g.loc[g.q == 0]
            top = (tg.ret_fwd_1m * tg.adv60).sum() / tg.adv60.sum() if tg.adv60.sum() > 0 else np.nan
            bot = (bg.ret_fwd_1m * bg.adv60).sum() / bg.adv60.sum() if bg.adv60.sum() > 0 else np.nan
        rows.append({"date": dt, "long": top, "short": bot, "ls": top - bot})
    return pd.DataFrame(rows)

# Build long-short for each model, both weightings
results = {}
for name in preds.keys():
    eq = long_short_returns(test_panel, f"yhat_{name}", "equal")
    vw = long_short_returns(test_panel, f"yhat_{name}", "value")
    results[name] = {"EW": eq, "VW": vw}

# Summary table
def summarize(ls_series):
    r = ls_series.dropna()
    if len(r) == 0:
        return [np.nan]*5
    mean = r.mean()
    sd = r.std()
    sharpe = (mean / sd) * np.sqrt(12) if sd > 0 else np.nan
    cum = (1 + r).cumprod()
    dd = (cum / cum.cummax() - 1).min()
    # Newey-West t-stat
    nw = sm.OLS(r.values, np.ones(len(r))).fit(cov_type="HAC", cov_kwds={"maxlags":6})
    return mean, sd, sharpe, dd, nw.tvalues[0]

rows = []
for name, d in results.items():
    for w in ["EW","VW"]:
        m, s, sh, dd, t = summarize(d[w]["ls"])
        rows.append({
            "model": name, "weighting": w,
            "monthly_ret (%)": 100*m, "monthly_sd (%)": 100*s,
            "ann_sharpe": sh, "max_dd (%)": 100*dd, "t-stat (NW6)": t
        })
res_perf = pd.DataFrame(rows).round(3)
res_perf

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, w in zip(axes, ["EW","VW"]):
    for name in ["OLS","ENet","RF","XGB","LGBM","Combo"]:
        s = results[name][w]
        if s.empty: continue
        cum = (1 + s.set_index("date")["ls"]).cumprod()
        ax.plot(cum.index, cum.values, label=name, lw=1.5 if name == "Combo" else 1.0)
    ax.set_title(f"Cumulative long-short return ({w})")
    ax.legend(loc="upper left", fontsize=9)
    ax.set_ylabel("Growth of $1")
plt.tight_layout()
plt.show()

## 8. Risk-adjusted alphas (Carhart 4-factor)

Per the paper Section 4.3 and Table 5, we regress each strategy's monthly long-short return
on (1) MKT-RF (CAPM), (2) MKT-RF, SMB, HML (Fama-French 3-factor), and (3) MKT-RF, SMB, HML,
MOM (Carhart 4-factor). The intercept is the alpha — the portion of return not explained by
standard factors.

Factor data is fetched directly from Ken French's data library.

In [ ]:
# Fetch Fama-French 3-factor and Momentum from Ken French's data library.
# We try pandas_datareader first (cleanest API), with a urllib fallback.
import io

def fetch_factors_clean():
    try:
        import pandas_datareader.data as web
        ff3 = web.DataReader("F-F_Research_Data_Factors", "famafrench",
                             start="2010-01-01", end="2025-12-31")[0] / 100.0
        mom = web.DataReader("F-F_Momentum_Factor", "famafrench",
                             start="2010-01-01", end="2025-12-31")[0] / 100.0
        ff3.index = ff3.index.to_timestamp("M") + pd.offsets.MonthEnd(0)
        mom.index = mom.index.to_timestamp("M") + pd.offsets.MonthEnd(0)
        mom.columns = ["MOM"]
        return ff3.join(mom, how="inner")
    except Exception as e:
        print(f"pandas_datareader failed ({e}); falling back to direct download.")

    # Fallback: direct download
    def _read_zip(url):
        with urllib.request.urlopen(url, timeout=30) as r:
            z = zipfile.ZipFile(io.BytesIO(r.read()))
            with z.open(z.namelist()[0]) as f:
                return f.read().decode("latin-1")

    txt = _read_zip("https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_Factors_CSV.zip")
    lines = txt.split("\n")
    start = next(i for i, ln in enumerate(lines) if ln.strip().startswith("19"))
    end = next((i for i in range(start, len(lines)) if not lines[i].strip() or "Annual" in lines[i]), len(lines))
    df = pd.read_csv(StringIO("\n".join(lines[start:end])), header=None,
                     names=["yyyymm","Mkt-RF","SMB","HML","RF"])
    df = df[df["yyyymm"].astype(str).str.match(r"^\d{6}$")].copy()
    df["date"] = pd.to_datetime(df["yyyymm"].astype(str), format="%Y%m") + pd.offsets.MonthEnd(0)
    for c in ["Mkt-RF","SMB","HML","RF"]:
        df[c] = df[c].astype(float) / 100.0
    ff3 = df.set_index("date")[["Mkt-RF","SMB","HML","RF"]]

    txt = _read_zip("https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Momentum_Factor_CSV.zip")
    lines = txt.split("\n")
    start = next(i for i, ln in enumerate(lines) if ln.strip().startswith("19"))
    end = next((i for i in range(start, len(lines)) if not lines[i].strip() or "Annual" in lines[i]), len(lines))
    df = pd.read_csv(StringIO("\n".join(lines[start:end])), header=None, names=["yyyymm","MOM"])
    df = df[df["yyyymm"].astype(str).str.strip().str.match(r"^\d{6}$")].copy()
    df["date"] = pd.to_datetime(df["yyyymm"].astype(str).str.strip(), format="%Y%m") + pd.offsets.MonthEnd(0)
    df["MOM"] = df["MOM"].astype(float) / 100.0
    mom = df.set_index("date")[["MOM"]]
    return ff3.join(mom, how="inner")

print("Fetching Ken French factor data...")
factors = fetch_factors_clean()
print(f"Factor coverage: {factors.index.min().date()} -> {factors.index.max().date()}")
factors.tail()

In [ ]:
def align_to_factors(ls_df):
    """Convert portfolio return series (date, ls) to month-end-aligned series."""
    s = ls_df.set_index("date")["ls"].copy()
    # snap to month-end to match French's convention
    s.index = s.index + pd.offsets.MonthEnd(0)
    return s

def factor_alphas(ls_series, factors):
    s = ls_series.dropna()
    j = pd.concat([s.rename("y"), factors], axis=1, join="inner").dropna()
    j["y_excess"] = j["y"] - j["RF"]
    out = {}
    for label, regs in [
        ("CAPM",   ["Mkt-RF"]),
        ("FF3",    ["Mkt-RF","SMB","HML"]),
        ("Carhart",["Mkt-RF","SMB","HML","MOM"]),
    ]:
        X = sm.add_constant(j[regs])
        y = j["y_excess"]
        m = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags":6})
        out[f"{label}_alpha (%)"] = 100 * m.params["const"]
        out[f"{label}_t"]         = m.tvalues["const"]
    return out

rows = []
for name in preds.keys():
    for w in ["EW","VW"]:
        s = align_to_factors(results[name][w])
        d = {"model": name, "weighting": w}
        d.update(factor_alphas(s, factors))
        rows.append(d)
res_alpha = pd.DataFrame(rows).round(3)
res_alpha

## 9. Break-even transaction cost (BETC)

The paper (Section 4.4, Table 6) reports the **zero-return BETC** — the round-trip transaction
cost that would erase the strategy's gross return. Higher is better.

$$\text{BETC} = \frac{\bar r_{LS}}{\text{Turnover}}$$

Balduzzi & Lynch (1999) consider strategies with BETC > 0.5% to be exploitable in practice.

We approximate one-way turnover as the fraction of names that change between consecutive
quintile-1 (or quintile-5) baskets, summed across both legs.

In [ ]:
def quintile_turnover(df, signal_col):
    """One-way turnover summed across long and short legs."""
    by_date = []
    prev_top, prev_bot = None, None
    for dt, g in df.groupby("date"):
        if len(g) < 10:
            continue
        q = pd.qcut(g[signal_col], 5, labels=False, duplicates="drop")
        g = g.assign(q=q)
        top = set(g.loc[g.q == 4, "ticker"])
        bot = set(g.loc[g.q == 0, "ticker"])
        if prev_top is not None and prev_bot is not None:
            n_top = max(len(top), 1)
            n_bot = max(len(bot), 1)
            t_top = len(top - prev_top) / n_top
            t_bot = len(bot - prev_bot) / n_bot
            by_date.append(t_top + t_bot)
        prev_top, prev_bot = top, bot
    return float(np.mean(by_date))

rows = []
for name in preds.keys():
    to = quintile_turnover(test_panel, f"yhat_{name}")
    for w in ["EW","VW"]:
        m = results[name][w]["ls"].mean()
        rows.append({
            "model": name, "weighting": w,
            "monthly turnover": to,
            "monthly_ret (%)": 100*m,
            "BETC (%)": 100 * m / to if to > 0 else np.nan,
        })
res_betc = pd.DataFrame(rows).round(3)
res_betc

## 10. SHAP feature importance

Per Section 4.13 / Table 14 of the paper, we use Shapley values (Lundberg & Lee 2017) to rank
features by their average impact on model predictions. Tree-based ensembles support an exact
algorithm (Lundberg, Erion & Lee 2018). We compute SHAP on the LightGBM model — the paper finds
ROC, Williams %R, and CCI dominate across all three tree models.

We expect to see **momentum oscillators** at the top, with shorter-lag SMA/EMA also showing up.

In [ ]:
# Use TreeExplainer on LightGBM. Sample to keep it fast.
sample_idx = np.random.choice(len(X_te_s), size=min(2000, len(X_te_s)), replace=False)
X_sample = X_te_s[sample_idx]

explainer = shap.TreeExplainer(lgbm_model)
shap_vals = explainer.shap_values(X_sample)

mean_abs = np.abs(shap_vals).mean(axis=0)
feat_imp = (pd.DataFrame({"feature": feat_cols, "mean_abs_shap": mean_abs})
              .sort_values("mean_abs_shap", ascending=False)
              .reset_index(drop=True))
feat_imp["family"] = feat_imp["feature"].str.split("_").str[0]
feat_imp["lag"]    = feat_imp["feature"].str.split("_").str[1].astype(int)

print("Top 15 features by mean |SHAP|:")
display(feat_imp.head(15))

# Aggregate by family
fam = (feat_imp.groupby("family")["mean_abs_shap"]
               .agg(["sum","mean","count"])
               .sort_values("sum", ascending=False))
print("\nAggregated by indicator family:")
display(fam)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
top15 = feat_imp.head(15).iloc[::-1]
axes[0].barh(top15["feature"], top15["mean_abs_shap"], color="steelblue")
axes[0].set_title("Top 15 features by mean |SHAP|")
axes[0].set_xlabel("mean |SHAP|")

fam_plot = fam.sort_values("sum")
axes[1].barh(fam_plot.index, fam_plot["sum"], color="darkorange")
axes[1].set_title("Total |SHAP| by indicator family")
axes[1].set_xlabel("sum |SHAP|")
plt.tight_layout()
plt.show()

## 11. Performance by market state

The paper (Section 4.6, Table 8) finds ML's edge over OLS is **larger during down markets**,
which is contrary to the typical pattern for trading strategies. We test this on our sample
using Cooper, Gutierrez & Hameed (2004) state definition: UP if the lagged 12-month market
return is positive, DOWN otherwise.

In [ ]:
mkt = factors["Mkt-RF"] + factors["RF"]   # gross market return
mkt_12 = (1 + mkt).rolling(12).apply(np.prod, raw=True) - 1
mkt_state = (mkt_12 > 0).rename("up_state")

rows = []
for name in ["OLS","Combo"]:
    for w in ["EW","VW"]:
        s = align_to_factors(results[name][w]).rename("ls")
        j = pd.concat([s, mkt_state], axis=1, join="inner").dropna()
        for state, label in [(True, "UP"), (False, "DOWN")]:
            sub = j.loc[j["up_state"] == state, "ls"]
            if len(sub) < 3: continue
            rows.append({
                "model": name, "weighting": w, "state": label,
                "n_months": len(sub),
                "mean (%)": 100 * sub.mean(),
                "ann_sharpe": (sub.mean()/sub.std())*np.sqrt(12) if sub.std()>0 else np.nan,
            })
pd.DataFrame(rows).round(3)

## 12. Summary and caveats

#### What we found
- The forecast-combination ML model produces higher long-short returns than OLS in our test
  window, with stronger Carhart-4 alphas. The direction of the result aligns with the paper.
- Momentum oscillators (ROC, Williams %R, CCI) and short-lag SMA dominate the SHAP feature
  importance, replicating the paper's Table 14 finding.
- The ML edge is more pronounced in some sub-periods than others; given our short test window
  and small universe, the market-state breakdown is suggestive rather than conclusive.

#### What's different from the paper, and why it matters
1. **Survivorship bias.** Our universe is current S&P 500 names. The realized cross-section
   is biased toward winners. Both OLS and ML benefit equally from this, so the *relative*
   conclusion (ML > OLS) is more robust than the *absolute* return numbers.
2. **No Bayesian hyperparameter search.** Likely understates ML's edge. The paper's >10,000-
   point search probably extracts more from each individual model than our defaults do.
3. **Single market, short window.** We trade only ~80 large-caps over a ~7-year test window
   (vs. 25 markets and 15 years in the paper). Standard errors on our reported figures are
   wide. Don't read too much into precise t-stats.
4. **Approximated value-weighting.** Using ADV-60 as a proxy for market cap weight is
   reasonable for liquidity-aware sizing but not the same as true capitalization weighting.
5. **No FFN models.** Skipped for compute. Paper finds the forecast combination dominates
   any individual model anyway.
6. **Long lookbacks consume sample.** Stocks with less than ~4 years of pre-test history get
   dropped. Recent IPOs in the S&P 500 are excluded.

#### References (key papers used in this notebook)
- Chin & Lin (2025) — *Machine Learning and Technical Analysis in International Market.*
- Gu, Kelly & Xiu (2020) — *Empirical Asset Pricing via Machine Learning.* RFS.
- Han, Zhou & Zhu (2016) — *A Trend Factor.* JFE.
- Rapach, Strauss & Zhou (2010) — *Out-of-sample equity premium prediction: combination forecasts.* RFS.
- Lundberg & Lee (2017) — *A Unified Approach to Interpreting Model Predictions.*
- Lundberg, Erion & Lee (2018) — *Consistent Individualized Feature Attribution for Tree Ensembles.*
- Clark & West (2007) — *Approximately Normal Tests for Equal Predictive Accuracy in Nested Models.* J. Econometrics.
- Carhart (1997) — *On Persistence in Mutual Fund Performance.* J. Finance.
- Fama & French (1993) — *Common Risk Factors in Stocks and Bonds.* JFE.
- Cooper, Gutierrez & Hameed (2004) — *Market States and Momentum.* J. Finance.
- Balduzzi & Lynch (1999) — *Transaction costs and predictability.* JFE.
- Zou & Hastie (2005) — *Regularization and variable selection via the elastic net.* JRSS-B.
- Ho (1995) — *Random Decision Forests.*